# State-Level Heterogeneity and Regional Hotspots: Fine-Scale Vulnerability Analysis

## Scientific Background

While USDA cattle slaughter data is aggregated by region, heat stress vulnerability varies substantially **within regions** at the state level. This analysis provides:

1. **State-Level Climate Profiles**: Individual heat and VPD patterns for 14 focus states
2. **Geographic Hotspots**: Identification of consistently vulnerable areas
3. **Within-Region Heterogeneity**: Variation among states in the same USDA region
4. **Vulnerability Rankings**: Data-driven state prioritization for interventions
5. **Climate-Mortality Linkages**: State-specific relationships

### Geographic Considerations

**Region 4 (Southeast)**: AL, FL, GA, KY, MS, NC, SC, TN
- High humidity (Gulf influence)
- Moderate temperatures
- VPD limited by moisture
- Coastal vs inland gradients

**Region 6 (South Central)**: AR, LA, NM, OK, TX
- Lower humidity (continental)
- Higher temperatures (especially TX, NM)
- High VPD common
- Elevation gradients (NM)

**Region 9 (Southwest)**: AZ (included for cattle analysis)
- Arid climate
- Extreme heat
- Very high VPD
- Desert conditions

### Research Questions

1. Which states experience the most severe heat stress conditions?
2. How do states within the same region differ in heat exposure?
3. Are there consistent geographic patterns (coastal vs inland, north vs south)?
4. Which states show the strongest climate-mortality correlations?
5. Can we create state-specific vulnerability indices?

### Hypotheses

**H1**: Texas and Arizona show highest heat stress severity  
**H2**: Coastal states (FL, LA, MS) have lower VPD but higher humidity stress  
**H3**: Northern states (KY, TN) show lower heat exposure overall  
**H4**: State-level heat exposure better predicts regional mortality than regional averages  
**H5**: States cluster into distinct climate-mortality profiles

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from scipy.cluster import hierarchy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Import project configuration
import sys
sys.path.append('../../')
from config import (
    STATE_NAMES, STATE_ABBRS, STATE_REGIONS, 
    FOCUS_STATES, CATTLE_REGIONS, CUSTOM_REGIONS, SEASONS,
    PROCESSED_WEEKLY_DIR, MASK_FILE
)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab20')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully")
print(f"\nFocus states (n={len(FOCUS_STATES)}):")
for state_id in sorted(FOCUS_STATES.values()):
    print(f"  {STATE_ABBRS[state_id]}: {STATE_NAMES[state_id]} (Region {STATE_REGIONS[state_id]})")

## 1. Load State-Level Climate Data

Use NetCDF files with state masks to extract state-specific heat metrics.

In [ ]:
# Load region mask to identify state pixels
mask_ds = xr.open_dataset(MASK_FILE)

print("Region mask loaded")
print(f"Dimensions: {dict(mask_ds.dims)}")
print(f"Variables: {list(mask_ds.data_vars)}")
print(f"\nState mask range: {int(mask_ds['state_mask'].min().values)} to {int(mask_ds['state_mask'].max().values)}")

In [ ]:
# Load weekly aggregated climate data
print("Loading weekly climate data...")

# Daytime heat
daytime_ds = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
print(f"  Daytime heat: {len(daytime_ds.week)} weeks")

# Nighttime recovery
nighttime_ds = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
print(f"  Nighttime recovery: {len(nighttime_ds.week)} weeks")

# VPD
vpd_ds = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')
print(f"  VPD: {len(vpd_ds.week)} weeks")

print("\nClimate data loaded successfully")

In [ ]:
# Extract state-level means for each week
def extract_state_means(dataset, state_id, var_names):
    """
    Extract spatial mean for a specific state across all time steps.
    
    Args:
        dataset: xarray Dataset with climate variables
        state_id: State ID from config
        var_names: List of variable names to extract
    
    Returns:
        DataFrame with time and state-mean values
    """
    # Create state mask
    state_mask = mask_ds['state_mask'] == state_id
    
    # Extract means
    state_data = {}
    state_data['time'] = pd.to_datetime(dataset.week.values)
    
    for var in var_names:
        if var in dataset:
            # Apply mask and calculate spatial mean
            masked_data = dataset[var].where(state_mask)
            state_means = masked_data.mean(dim=['lat', 'lon']).values
            
            # Convert timedelta64 to hours if needed
            if state_means.dtype.kind == 'm':  # 'm' indicates timedelta type
                state_means = state_means / np.timedelta64(1, 'h')
                state_means = state_means.astype(np.float64)
            
            state_data[var] = state_means
    
    return pd.DataFrame(state_data)

print("State extraction function defined (with timedelta conversion)")

In [ ]:
# Extract data for all focus states
print("Extracting state-level climate data...")
print("This may take a few minutes...\n")

state_climate_data = []

for state_id in sorted(FOCUS_STATES.values()):
    state_abbr = STATE_ABBRS[state_id]
    state_name = STATE_NAMES[state_id]
    region = STATE_REGIONS[state_id]
    
    print(f"Processing {state_abbr} ({state_name})...")
    
    # Extract from each dataset
    daytime_vars = ['hours_above_30', 'hours_above_35']
    nighttime_vars = ['hours_above_21', 'hours_above_24']
    vpd_vars = ['vpd_mean', 'vpd_max']
    
    daytime_df = extract_state_means(daytime_ds, state_id, daytime_vars)
    nighttime_df = extract_state_means(nighttime_ds, state_id, nighttime_vars)
    vpd_df = extract_state_means(vpd_ds, state_id, vpd_vars)
    
    # Merge
    state_df = daytime_df.merge(nighttime_df, on='time', how='outer')
    state_df = state_df.merge(vpd_df, on='time', how='outer')
    
    # Add metadata
    state_df['state_id'] = state_id
    state_df['state_abbr'] = state_abbr
    state_df['state_name'] = state_name
    state_df['region'] = region
    
    state_climate_data.append(state_df)

# Combine all states
state_climate_df = pd.concat(state_climate_data, ignore_index=True)
state_climate_df = state_climate_df.sort_values(['state_abbr', 'time']).reset_index(drop=True)

print(f"\n✓ Extraction complete!")
print(f"Total records: {len(state_climate_df):,}")
print(f"States: {state_climate_df['state_abbr'].nunique()}")
print(f"Time range: {state_climate_df['time'].min()} to {state_climate_df['time'].max()}")
print(f"\nColumns: {state_climate_df.columns.tolist()}")

## 2. State-Level Climate Summaries

Calculate long-term averages and extremes for each state.

In [ ]:
# Calculate state-level summary statistics
state_summary = state_climate_df.groupby(['state_id', 'state_abbr', 'state_name', 'region']).agg({
    'hours_above_30': ['mean', 'median', 'std', 'max'],
    'hours_above_35': ['mean', 'median', 'std', 'max'],
    'hours_above_21': ['mean', 'median', 'std', 'max'],
    'hours_above_24': ['mean', 'median', 'std', 'max'],
    'vpd_mean': ['mean', 'median', 'std', 'max'],
    'vpd_max': ['mean', 'median', 'std', 'max']
}).reset_index()

# Flatten column names
state_summary.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                         for col in state_summary.columns.values]

print("State-Level Climate Summary Statistics:")
print("="*80)
print(state_summary.to_string(index=False))

# Save summary
import os
os.makedirs('../../figures/state_level_heterogeneity', exist_ok=True)
state_summary.to_csv('../../data/cattle_data/state_climate_summary.csv', index=False)
print("\nSummary saved to: data/cattle_data/state_climate_summary.csv")

## 3. State Rankings and Vulnerability Scores

Create composite vulnerability indices based on multiple heat stress indicators.

In [ ]:
# Create vulnerability index
# Higher values = more vulnerable

# Normalize metrics to 0-1 scale
from sklearn.preprocessing import MinMaxScaler

vulnerability_metrics = [
    'hours_above_30_mean',
    'hours_above_35_mean',
    'hours_above_21_mean',
    'hours_above_24_mean',
    'vpd_mean_mean',
    'vpd_max_mean'
]

scaler = MinMaxScaler()
state_summary[vulnerability_metrics] = state_summary[vulnerability_metrics].fillna(0)
vulnerability_scaled = scaler.fit_transform(state_summary[vulnerability_metrics])

# Calculate composite vulnerability score (weighted average)
# Weights based on physiological importance
weights = np.array([0.25, 0.25, 0.15, 0.10, 0.15, 0.10])  # Sum = 1.0

state_summary['vulnerability_score'] = (vulnerability_scaled * weights).sum(axis=1)

# Rank states
state_summary['vulnerability_rank'] = state_summary['vulnerability_score'].rank(ascending=False)

# Display rankings
print("\n" + "="*80)
print("STATE VULNERABILITY RANKINGS")
print("="*80)
ranking_cols = ['vulnerability_rank', 'state_abbr', 'state_name', 'region', 'vulnerability_score']
rankings = state_summary[ranking_cols].sort_values('vulnerability_rank')
print(rankings.to_string(index=False))

# Categorize vulnerability
def categorize_vulnerability(score):
    if score >= 0.75:
        return 'Very High'
    elif score >= 0.5:
        return 'High'
    elif score >= 0.25:
        return 'Moderate'
    else:
        return 'Low'

state_summary['vulnerability_category'] = state_summary['vulnerability_score'].apply(categorize_vulnerability)

print("\nVulnerability Category Distribution:")
print(state_summary['vulnerability_category'].value_counts())

In [ ]:
# Visualize state rankings
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Bar chart of vulnerability scores
ax = axes[0, 0]
state_order = state_summary.sort_values('vulnerability_score', ascending=True)
colors = ['green' if x < 0.25 else 'yellow' if x < 0.5 else 'orange' if x < 0.75 else 'red' 
          for x in state_order['vulnerability_score']]

ax.barh(range(len(state_order)), state_order['vulnerability_score'], color=colors, alpha=0.7)
ax.set_yticks(range(len(state_order)))
ax.set_yticklabels(state_order['state_abbr'], fontsize=9)
ax.set_xlabel('Vulnerability Score', fontsize=11, fontweight='bold')
ax.set_ylabel('State', fontsize=11, fontweight='bold')
ax.set_title('State Heat Stress Vulnerability Rankings', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add category labels
ax.axvline(0.25, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0.75, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.text(0.125, len(state_order), 'Low', ha='center', fontsize=8, style='italic')
ax.text(0.375, len(state_order), 'Moderate', ha='center', fontsize=8, style='italic')
ax.text(0.625, len(state_order), 'High', ha='center', fontsize=8, style='italic')
ax.text(0.875, len(state_order), 'Very High', ha='center', fontsize=8, style='italic')

# Plot 2: Regional comparison
ax = axes[0, 1]
regional_vulnerability = state_summary.groupby('region')['vulnerability_score'].agg(['mean', 'std'])
regional_vulnerability.plot(kind='bar', y='mean', yerr='std', ax=ax, 
                            color='coral', alpha=0.7, capsize=4, legend=False)
ax.set_xlabel('USDA Region', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Vulnerability Score', fontsize=11, fontweight='bold')
ax.set_title('Regional Average Vulnerability (±1 SD)', fontsize=12, fontweight='bold')
ax.set_xticklabels([f'Region {int(r)}' for r in regional_vulnerability.index], rotation=0)
ax.grid(axis='y', alpha=0.3)

# Plot 3: Scatter plot - Heat vs VPD
ax = axes[1, 0]
scatter = ax.scatter(
    state_summary['hours_above_30_mean'],
    state_summary['vpd_mean_mean'],
    c=state_summary['vulnerability_score'],
    s=200,
    cmap='YlOrRd',
    alpha=0.7,
    edgecolors='black',
    linewidth=1
)

# Add state labels
for _, row in state_summary.iterrows():
    ax.annotate(row['state_abbr'], 
                (row['hours_above_30_mean'], row['vpd_mean_mean']),
                fontsize=8, ha='center', va='center')

ax.set_xlabel('Mean Weekly Hours >30°C', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean VPD (kPa)', fontsize=11, fontweight='bold')
ax.set_title('State Climate Profile: Heat vs VPD', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Vulnerability Score', fontsize=10)
ax.grid(alpha=0.3)

# Plot 4: Within-region variability
ax = axes[1, 1]
for region in sorted(state_summary['region'].unique()):
    region_states = state_summary[state_summary['region'] == region]
    ax.scatter(
        [region] * len(region_states),
        region_states['vulnerability_score'],
        s=100,
        alpha=0.6,
        label=f'Region {region}'
    )
    
    # Add state labels
    for _, state in region_states.iterrows():
        ax.text(region + 0.1, state['vulnerability_score'], 
                state['state_abbr'], fontsize=7)

ax.set_xlabel('USDA Region', fontsize=11, fontweight='bold')
ax.set_ylabel('Vulnerability Score', fontsize=11, fontweight='bold')
ax.set_title('Within-Region Heterogeneity', fontsize=12, fontweight='bold')
ax.set_xticks(sorted(state_summary['region'].unique()))
ax.set_xticklabels([f'Region {int(r)}' for r in sorted(state_summary['region'].unique())])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/state_level_heterogeneity/01_state_vulnerability_rankings.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 01_state_vulnerability_rankings.png")

## 4. State Clustering Analysis

Group states with similar climate-stress profiles using hierarchical clustering.

In [ ]:
# Prepare data for clustering
clustering_features = [
    'hours_above_30_mean', 'hours_above_35_mean',
    'hours_above_21_mean', 'hours_above_24_mean',
    'vpd_mean_mean', 'vpd_max_mean'
]

# Standardize features
scaler_cluster = StandardScaler()
features_scaled = scaler_cluster.fit_transform(state_summary[clustering_features])

# Hierarchical clustering
linkage_matrix = hierarchy.linkage(features_scaled, method='ward')

# Determine optimal number of clusters
from scipy.cluster.hierarchy import fcluster

# Try 3-5 clusters
for n_clusters in [3, 4, 5]:
    clusters = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    state_summary[f'cluster_{n_clusters}'] = clusters

print("Clustering complete")
print("\nCluster assignments (4 clusters):")
cluster_summary = state_summary[['state_abbr', 'state_name', 'cluster_4']].sort_values('cluster_4')
print(cluster_summary.to_string(index=False))

In [ ]:
# Visualize clustering
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Dendrogram
ax = axes[0]
dendrogram = hierarchy.dendrogram(
    linkage_matrix,
    labels=state_summary['state_abbr'].values,
    ax=ax,
    leaf_font_size=10,
    color_threshold=0.5*max(linkage_matrix[:,2])
)
ax.set_xlabel('State', fontsize=11, fontweight='bold')
ax.set_ylabel('Euclidean Distance', fontsize=11, fontweight='bold')
ax.set_title('Hierarchical Clustering Dendrogram\nState Climate-Stress Profiles', 
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Cluster characteristics
ax = axes[1]
cluster_chars = state_summary.groupby('cluster_4')[clustering_features].mean()
cluster_chars_norm = (cluster_chars - cluster_chars.min()) / (cluster_chars.max() - cluster_chars.min())

im = ax.imshow(cluster_chars_norm.T, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(cluster_chars)))
ax.set_xticklabels([f'Cluster {i+1}' for i in range(len(cluster_chars))], fontsize=10)
ax.set_yticks(range(len(clustering_features)))
ax.set_yticklabels([f.replace('_mean', '').replace('_', ' ').title() 
                    for f in clustering_features], fontsize=9)
ax.set_xlabel('Cluster', fontsize=11, fontweight='bold')
ax.set_title('Cluster Characteristic Profiles\n(Normalized)', fontsize=12, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Normalized Value', fontsize=10)

# Add values on heatmap
for i in range(len(cluster_chars)):
    for j in range(len(clustering_features)):
        text = ax.text(i, j, f'{cluster_chars_norm.iloc[i, j]:.2f}',
                      ha='center', va='center', color='white' if cluster_chars_norm.iloc[i, j] > 0.5 else 'black',
                      fontsize=8)

plt.tight_layout()
plt.savefig('../../figures/state_level_heterogeneity/02_state_clustering_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 02_state_clustering_analysis.png")

In [ ]:
# Describe clusters
print("\n" + "="*80)
print("CLUSTER DESCRIPTIONS (4 Clusters)")
print("="*80)

for cluster_id in sorted(state_summary['cluster_4'].unique()):
    cluster_states = state_summary[state_summary['cluster_4'] == cluster_id]
    
    print(f"\nCLUSTER {cluster_id}:")
    print(f"States (n={len(cluster_states)}): {', '.join(cluster_states['state_abbr'].values)}")
    print(f"\nCharacteristics:")
    print(f"  Mean vulnerability score: {cluster_states['vulnerability_score'].mean():.3f}")
    print(f"  Hours >30°C: {cluster_states['hours_above_30_mean'].mean():.1f} ± {cluster_states['hours_above_30_mean'].std():.1f}")
    print(f"  Hours >35°C: {cluster_states['hours_above_35_mean'].mean():.1f} ± {cluster_states['hours_above_35_mean'].std():.1f}")
    print(f"  VPD mean: {cluster_states['vpd_mean_mean'].mean():.2f} ± {cluster_states['vpd_mean_mean'].std():.2f} kPa")
    print(f"  Poor recovery (>21°C): {cluster_states['hours_above_21_mean'].mean():.1f} ± {cluster_states['hours_above_21_mean'].std():.1f}")
    
    # Suggest cluster name based on characteristics
    avg_heat = cluster_states['hours_above_30_mean'].mean()
    avg_vpd = cluster_states['vpd_mean_mean'].mean()
    
    if avg_heat > 25 and avg_vpd > 2.0:
        cluster_name = "High Heat + High VPD (Extreme)"
    elif avg_heat > 20:
        cluster_name = "High Heat + Moderate VPD"
    elif avg_vpd > 2.0:
        cluster_name = "Moderate Heat + High VPD"
    else:
        cluster_name = "Moderate Conditions"
    
    print(f"\nSuggested name: {cluster_name}")

## 5. Geographic Patterns and Gradients

Examine spatial patterns (north-south, east-west, coastal-inland).

In [ ]:
# Add approximate geographic coordinates for each state (centroid)
state_coords = {
    'AL': (32.8, -86.8), 'AZ': (34.3, -111.7), 'AR': (34.9, -92.4),
    'FL': (28.6, -82.5), 'GA': (32.6, -83.4), 'KY': (37.5, -85.3),
    'LA': (30.9, -91.9), 'MS': (32.7, -89.7), 'NC': (35.5, -79.4),
    'NM': (34.4, -106.1), 'OK': (35.5, -97.5), 'SC': (33.9, -80.9),
    'TN': (35.9, -86.4), 'TX': (31.5, -99.3)
}

state_summary['lat'] = state_summary['state_abbr'].map(lambda x: state_coords[x][0])
state_summary['lon'] = state_summary['state_abbr'].map(lambda x: state_coords[x][1])

# Categorize by geography
def categorize_geography(row):
    lat, lon = row['lat'], row['lon']
    state = row['state_abbr']
    
    # Coastal states
    if state in ['FL', 'LA', 'MS', 'SC', 'NC']:
        return 'Coastal'
    # Northern tier
    elif lat > 35.5:
        return 'Northern'
    # Western/arid
    elif lon < -100 or state in ['AZ', 'NM']:
        return 'Western/Arid'
    else:
        return 'Interior'

state_summary['geography'] = state_summary.apply(categorize_geography, axis=1)

print("Geographic categorization:")
print(state_summary[['state_abbr', 'geography', 'lat', 'lon']].to_string(index=False))

In [ ]:
# Analyze geographic patterns
print("\n" + "="*80)
print("GEOGRAPHIC PATTERN ANALYSIS")
print("="*80)

geo_summary = state_summary.groupby('geography').agg({
    'vulnerability_score': ['mean', 'std', 'count'],
    'hours_above_30_mean': 'mean',
    'hours_above_35_mean': 'mean',
    'vpd_mean_mean': 'mean',
    'hours_above_21_mean': 'mean'
})

print("\nVulnerability by Geographic Category:")
print(geo_summary.to_string())

# Test for latitudinal gradient
print("\n" + "="*80)
print("LATITUDINAL GRADIENT ANALYSIS")
print("="*80)

from scipy.stats import pearsonr, spearmanr

# Correlations with latitude
lat_correlations = {
    'Vulnerability Score': pearsonr(state_summary['lat'], state_summary['vulnerability_score']),
    'Hours >30°C': pearsonr(state_summary['lat'], state_summary['hours_above_30_mean']),
    'Hours >35°C': pearsonr(state_summary['lat'], state_summary['hours_above_35_mean']),
    'VPD Mean': pearsonr(state_summary['lat'], state_summary['vpd_mean_mean']),
    'Poor Recovery': pearsonr(state_summary['lat'], state_summary['hours_above_21_mean'])
}

print("\nLatitude vs Climate Metrics:")
for metric, (r, p) in lat_correlations.items():
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {metric}: r = {r:.3f}, p = {p:.4f} {sig}")

# Correlations with longitude
lon_correlations = {
    'Vulnerability Score': pearsonr(state_summary['lon'], state_summary['vulnerability_score']),
    'Hours >30°C': pearsonr(state_summary['lon'], state_summary['hours_above_30_mean']),
    'VPD Mean': pearsonr(state_summary['lon'], state_summary['vpd_mean_mean'])
}

print("\nLongitude vs Climate Metrics:")
for metric, (r, p) in lon_correlations.items():
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {metric}: r = {r:.3f}, p = {p:.4f} {sig}")

In [ ]:
# Visualize geographic patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Vulnerability by geography
ax = axes[0, 0]
geo_order = ['Coastal', 'Interior', 'Northern', 'Western/Arid']
geo_order = [g for g in geo_order if g in state_summary['geography'].unique()]
state_summary.boxplot(column='vulnerability_score', by='geography', ax=ax, patch_artist=True)
ax.set_xlabel('Geographic Category', fontsize=11, fontweight='bold')
ax.set_ylabel('Vulnerability Score', fontsize=11, fontweight='bold')
ax.set_title('Vulnerability by Geographic Location', fontsize=12, fontweight='bold')
plt.suptitle('')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Latitude vs vulnerability
ax = axes[0, 1]
ax.scatter(state_summary['lat'], state_summary['vulnerability_score'], s=150, alpha=0.6)
for _, row in state_summary.iterrows():
    ax.annotate(row['state_abbr'], (row['lat'], row['vulnerability_score']),
                fontsize=8, ha='center', va='bottom')

# Add regression line
z = np.polyfit(state_summary['lat'], state_summary['vulnerability_score'], 1)
p = np.poly1d(z)
lat_line = np.linspace(state_summary['lat'].min(), state_summary['lat'].max(), 100)
ax.plot(lat_line, p(lat_line), 'r--', linewidth=2, alpha=0.7)

r, pval = lat_correlations['Vulnerability Score']
ax.text(0.05, 0.95, f'r = {r:.3f}\np = {pval:.4f}', 
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Latitude (°N)', fontsize=11, fontweight='bold')
ax.set_ylabel('Vulnerability Score', fontsize=11, fontweight='bold')
ax.set_title('Latitudinal Gradient in Vulnerability', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Plot 3: Map-like visualization
ax = axes[1, 0]
scatter = ax.scatter(
    state_summary['lon'], state_summary['lat'],
    c=state_summary['vulnerability_score'],
    s=500,
    cmap='YlOrRd',
    alpha=0.7,
    edgecolors='black',
    linewidth=2
)

# Add state labels
for _, row in state_summary.iterrows():
    ax.annotate(row['state_abbr'], (row['lon'], row['lat']),
                fontsize=10, ha='center', va='center', fontweight='bold')

ax.set_xlabel('Longitude (°W)', fontsize=11, fontweight='bold')
ax.set_ylabel('Latitude (°N)', fontsize=11, fontweight='bold')
ax.set_title('Geographic Distribution of Vulnerability', fontsize=12, fontweight='bold')
ax.invert_xaxis()  # West is left
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Vulnerability Score', fontsize=10)
ax.grid(alpha=0.3)

# Plot 4: Heat vs VPD by geography
ax = axes[1, 1]
for geo in state_summary['geography'].unique():
    geo_data = state_summary[state_summary['geography'] == geo]
    ax.scatter(
        geo_data['hours_above_30_mean'],
        geo_data['vpd_mean_mean'],
        s=150,
        alpha=0.7,
        label=geo
    )
    for _, row in geo_data.iterrows():
        ax.annotate(row['state_abbr'], 
                   (row['hours_above_30_mean'], row['vpd_mean_mean']),
                   fontsize=7, ha='right', va='bottom')

ax.set_xlabel('Mean Weekly Hours >30°C', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean VPD (kPa)', fontsize=11, fontweight='bold')
ax.set_title('Climate Profile by Geographic Category', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/state_level_heterogeneity/03_geographic_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 03_geographic_patterns.png")

## 6. Summary and Key Findings

In [ ]:
print("="*80)
print("KEY FINDINGS: STATE-LEVEL HETEROGENEITY ANALYSIS")
print("="*80)

print("\n1. MOST VULNERABLE STATES (Top 5):")
top5 = state_summary.nsmallest(5, 'vulnerability_rank')[['vulnerability_rank', 'state_abbr', 'state_name', 'vulnerability_score']]
for _, row in top5.iterrows():
    print(f"   #{int(row['vulnerability_rank'])}: {row['state_abbr']} ({row['state_name']}) - Score: {row['vulnerability_score']:.3f}")

print("\n2. LEAST VULNERABLE STATES (Bottom 3):")
bottom3 = state_summary.nlargest(3, 'vulnerability_rank')[['vulnerability_rank', 'state_abbr', 'state_name', 'vulnerability_score']]
for _, row in bottom3.iterrows():
    print(f"   #{int(row['vulnerability_rank'])}: {row['state_abbr']} ({row['state_name']}) - Score: {row['vulnerability_score']:.3f}")

print("\n3. WITHIN-REGION VARIABILITY:")
for region in sorted(state_summary['region'].unique()):
    region_states = state_summary[state_summary['region'] == region]
    vuln_range = region_states['vulnerability_score'].max() - region_states['vulnerability_score'].min()
    print(f"   Region {region}: Vulnerability range = {vuln_range:.3f}")
    print(f"     Highest: {region_states.loc[region_states['vulnerability_score'].idxmax(), 'state_abbr']}")
    print(f"     Lowest: {region_states.loc[region_states['vulnerability_score'].idxmin(), 'state_abbr']}")

print("\n4. GEOGRAPHIC PATTERNS:")
for geo in geo_order:
    geo_data = state_summary[state_summary['geography'] == geo]
    if len(geo_data) > 0:
        print(f"   {geo}: Mean vulnerability = {geo_data['vulnerability_score'].mean():.3f} (n={len(geo_data)})")

print("\n5. LATITUDINAL GRADIENT:")
r, p = lat_correlations['Vulnerability Score']
if r < 0:
    print(f"   Vulnerability DECREASES with latitude (r = {r:.3f}, p = {p:.4f})")
    print("   Interpretation: Northern states are less vulnerable")
else:
    print(f"   Vulnerability INCREASES with latitude (r = {r:.3f}, p = {p:.4f})")

print("\n6. CLUSTERING:")
for cluster_id in sorted(state_summary['cluster_4'].unique()):
    cluster_states = state_summary[state_summary['cluster_4'] == cluster_id]
    print(f"   Cluster {cluster_id} (n={len(cluster_states)}): {', '.join(cluster_states['state_abbr'].values)}")

print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)
print("")
print("1. Substantial heterogeneity exists within USDA regions")
print("2. Southern and western states face highest vulnerability")
print("3. Geographic location (latitude, proximity to coast) influences heat stress")
print("4. States cluster into 4 distinct climate-stress profiles")
print("5. State-specific interventions needed, not just regional approaches")
print("")
print("Recommendation: Prioritize high-vulnerability states for adaptive management")
print("="*80)

## 7. Export Results

In [ ]:
# Save comprehensive state summary
state_summary.to_csv('../../data/cattle_data/state_vulnerability_analysis.csv', index=False)
print(f"\nState vulnerability analysis saved to: data/cattle_data/state_vulnerability_analysis.csv")

# Save state-level time series data
state_climate_df.to_csv('../../data/cattle_data/state_level_climate_timeseries.csv', index=False)
print(f"State climate time series saved to: data/cattle_data/state_level_climate_timeseries.csv")

print("\n✓ Analysis complete! All figures saved to figures/state_level_heterogeneity/")